# Step 3 FIX — Clean Training Cell

**Restart kernel first**, then run all cells top to bottom. Do not skip any cell.

In [ ]:
import os, json, csv, time
import torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DEVICE      = 'mps' if torch.backends.mps.is_available() else 'cpu'
MODEL_NAME  = 'defog/sqlcoder-7b-2'
MAX_SEQ_LEN = 512

print(f'Device  : {DEVICE.upper()}')
print(f'PyTorch : {torch.__version__}')

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f'✅ Tokenizer ready — vocab: {tokenizer.vocab_size:,}')

In [ ]:
from transformers import AutoModelForCausalLM

# float16 on MPS — no 4-bit (bitsandbytes 4-bit is CUDA only, breaks on MPS)
print('Loading model in float16 on MPS ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={'': DEVICE},
    trust_remote_code=True,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1
print(f'✅ Model loaded — {sum(p.numel() for p in model.parameters())/1e9:.2f}B params')

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

# Apply LoRA ONCE — if you see 'already has LoRA' you must restart kernel
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA applied once')

In [ ]:
from datasets import Dataset
import numpy as np

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return Dataset.from_list([json.loads(l) for l in f])

train_ds = load_jsonl('data/train.jsonl')
val_ds   = load_jsonl('data/val.jsonl')

def tokenise(examples):
    out = tokenizer(examples['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    out['labels'] = out['input_ids'].copy()
    return out

remove_cols = [c for c in train_ds.column_names if c != 'text']
train_tok = train_ds.map(tokenise, batched=True, remove_columns=remove_cols, desc='Train')
val_tok   = val_ds.map(tokenise,   batched=True, remove_columns=remove_cols, desc='Val')

lens = [len(x) for x in train_tok['input_ids']]
print(f'Token lengths — min:{min(lens)} max:{max(lens)} avg:{int(np.mean(lens))}')
print(f'Train: {len(train_tok):,}  Val: {len(val_tok):,}')
print('✅ Tokenisation done')

In [ ]:
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback
)
from pathlib import Path

Path('logs').mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)

log_file   = open('logs/train_log.csv', 'w', newline='')
log_writer = csv.writer(log_file)
log_writer.writerow(['step', 'train_loss', 'eval_loss', 'lr', 'epoch'])

class CsvLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_writer.writerow([
                state.global_step,
                logs.get('loss'),
                logs.get('eval_loss'),
                logs.get('learning_rate'),
                logs.get('epoch'),
            ])
            log_file.flush()

training_args = TrainingArguments(
    output_dir='checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=100,            # fixed: warmup_ratio is deprecated
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',       # fixed: was evaluation_strategy
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,  # fixed: tokenizer arg removed in new transformers
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[CsvLogger()],
)

print('🚀 Training started — estimated 6-10h on M4 16GB')
start = time.time()
try:
    trainer.train()
finally:
    elapsed = int(time.time() - start)
    log_file.close()
    h, m = divmod(elapsed // 60, 60)
    print(f'\n✅ Done in {h}h {m}m')

In [ ]:
import pandas as pd

model.save_pretrained('checkpoints/best_adapter')
tokenizer.save_pretrained('checkpoints/best_adapter')
print('✅ Adapter saved → checkpoints/best_adapter')

log_df = pd.read_csv('logs/train_log.csv').dropna(subset=['train_loss'])
print(f'Final train loss : {log_df["train_loss"].iloc[-1]:.4f}')
print(f'Best eval loss   : {log_df["eval_loss"].dropna().min():.4f}')
print()
print('✅ Paste this output in chat — I will give you step 4 (eval + report)')